# Projection and Predicate pushdown in Apache Parquet using Spark


In [1]:
%run_nb spark-start

Args: Namespace(data_format='none', port_offset=2) - unknown_args: []
Spark version: 4.1.3, Driver memory: 16g, Executor memory: 8g, Service: jupyter-spark-4.1, Data format: None
Spark packages: 
Spark extensions: 
Spark catalog configs: {}
spark.sql.shuffle.partitions: 200
spark.sparkContext.master: local[2]


Loading ITables v2.9.1 from the internet... (need help?)
🔒ⓘcatalog
spark_catalog


Version,4.1.3
Master,local[2]
AppName,main


               total        used        free      shared  buff/cache   available
Mem:            62Gi        20Gi       7.4Gi       1.2Gi        36Gi        42Gi
Swap:          8.0Gi       2.4Mi       8.0Gi


In [6]:
path="/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset"
!ls -la {path}

total 5799848
drwxr-xr-x. 1 jovyan users        332 Jul 24 14:32  .
drwxr-xr-x. 1 jovyan  1000         62 Jul 15 20:37  ..
drwxr-xr-x. 1 jovyan users         16 Jul 15 20:39  .complete
-rw-r--r--. 1 jovyan users       5356 Jul 15 20:38  female_coaches.csv
-rw-r--r--. 1 jovyan users    1685124 Jul 15 20:38 'female_players (legacy).csv'
-rw-r--r--. 1 jovyan users   94212088 Jul 15 20:38  female_players.csv
-rw-r--r--. 1 jovyan users    2214250 Jul 15 20:38  female_teams.csv
-rw-r--r--. 1 jovyan users     132879 Jul 15 20:38  male_coaches.csv
-rw-r--r--. 1 jovyan users   90933390 Jul 15 20:38 'male_players (legacy).csv'
-rw-r--r--. 1 jovyan users 5637100640 Jul 15 20:39  male_players.csv
-rw-r--r--. 1 jovyan users  112744779 Jul 15 20:39  male_teams.csv
drwxr-xr-x. 1 jovyan users         24 Jul 24 14:32  parquet
time: 113 ms (started: 2026-07-27 21:05:28 +00:00)


In [7]:
csv_file = Path(path) / "male_players.csv"
parquet_output = Path(path) / "parquet" / "male_players"

time: 569 μs (started: 2026-07-27 21:05:28 +00:00)


In [8]:
%load_ext autotime

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 292 μs (started: 2026-07-27 21:05:29 +00:00)


In [9]:
if not parquet_output.is_dir():
    print(f"Convert CSV to parquet")
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(str(csv_file))
    )

    df.write.mode("overwrite").parquet(str(parquet_output))
 
else:
    print(f"Read parquet")
    
    df = (
        spark.read
        .parquet(str(parquet_output))
    )      

Read parquet
time: 416 ms (started: 2026-07-27 21:05:29 +00:00)


In [10]:
!echo "Number of files: $(find {path}/parquet/male_players/*.parquet | wc -l)"
!echo "Files: $(ls {path}/parquet/male_players/*.parquet)"

Number of files: 42
Files: /home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parquet/male_players/part-00000-f15470c2-dc6d-416d-b024-edcc43eb8f76-c000.snappy.parquet
/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parquet/male_players/part-00001-f15470c2-dc6d-416d-b024-edcc43eb8f76-c000.snappy.parquet
/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parquet/male_players/part-00002-f15470c2-dc6d-416d-b024-edcc43eb8f76-c000.snappy.parquet
/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parquet/male_players/part-00003-f15470c2-dc6d-416d-b024-edcc43eb8f76-c000.snappy.parquet
/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parquet/male_players/part-00004-f15470c2-dc6d-416d-b024-edcc43eb8f76-c000.snappy.parquet
/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parque

In [11]:
humanize.intword(df.count())

'10.0 million'

time: 500 ms (started: 2026-07-27 21:05:35 +00:00)


In [12]:
df.explain()

== Physical Plan ==
FileScan parquet [player_id#4,player_url#5,fifa_version#6,fifa_update#7,fifa_update_date#8,short_name#9,long_name#10,player_positions#11,overall#12,potential#13,value_eur#14,wage_eur#15,age#16,dob#17,height_cm#18,weight_kg#19,league_id#20,league_name#21,league_level#22,club_team_id#23,club_name#24,club_position#25,club_jersey_number#26,club_loaned_from#27,club_joined_date#28,... 85 more fields] Batched: false, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-co..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<player_id:int,player_url:string,fifa_version:int,fifa_update:int,fifa_update_date:date,sho...


time: 8.95 ms (started: 2026-07-27 21:05:37 +00:00)


# Predicate pushdown

In [13]:
df_all = spark.read.parquet(str(parquet_output))

df_select = (
    spark.read
    .parquet(str(parquet_output))
    .select("value_eur")
)

time: 130 ms (started: 2026-07-27 21:05:40 +00:00)


In [14]:
df_select_filter = (
    spark.read
    .parquet(str(parquet_output))
    .filter(F.col("value_eur") > 1000000)
    .select("value_eur")
)

time: 71.1 ms (started: 2026-07-27 21:05:41 +00:00)


In [15]:
humanize.intword(df_select_filter.count())

'3.7 million'

time: 431 ms (started: 2026-07-27 21:05:42 +00:00)


In [16]:
humanize.intword(df_select.count())

'10.0 million'

time: 140 ms (started: 2026-07-27 21:05:42 +00:00)


In [17]:
viewdf(df_select_filter, limit=5)

Loading ITables v2.9.1 from the internet... (need help?)
🔒ⓘvalue_eur
103500000
63000000
111000000
132000000
129000000


time: 81.5 ms (started: 2026-07-27 21:05:44 +00:00)


# Physical plan
## See "PushedFilters:"

In [18]:
df_select_filter.explain("formatted")

== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [value_eur#459]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parquet/male_players]
PushedFilters: [IsNotNull(value_eur), GreaterThan(value_eur,1000000)]
ReadSchema: struct<value_eur:int>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [value_eur#459]

(3) Filter [codegen id : 1]
Input [1]: [value_eur#459]
Condition : (isnotnull(value_eur#459) AND (value_eur#459 > 1000000))


time: 8.41 ms (started: 2026-07-27 21:05:46 +00:00)


In [19]:
df_select.explain("formatted")

== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [value_eur#348]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parquet/male_players]
ReadSchema: struct<value_eur:int>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [value_eur#348]


time: 5.01 ms (started: 2026-07-27 21:05:47 +00:00)


In [20]:
humanize.intword(df_select_filter.count())

'3.7 million'

time: 199 ms (started: 2026-07-27 21:05:47 +00:00)


In [21]:
humanize.intword(df_select.count())

'10.0 million'

time: 95.3 ms (started: 2026-07-27 21:05:48 +00:00)


In [22]:
def benchmark(label, query):
    # Run once to reduce JVM / planning noise
    query.collect()

    start = time.perf_counter()
    result = query.collect()
    elapsed = time.perf_counter() - start

    print(f"{label}: {elapsed:.3f} s")
    #return result

time: 196 μs (started: 2026-07-27 21:05:49 +00:00)


In [23]:
benchmark("Only value_eur", df_select)

Only value_eur: 10.174 s
time: 21.5 s (started: 2026-07-27 21:05:49 +00:00)


In [24]:
df_select_filter.explain("simple")

== Physical Plan ==
*(1) Filter (isnotnull(value_eur#459) AND (value_eur#459 > 1000000))
+- *(1) ColumnarToRow
   +- FileScan parquet [value_eur#459] Batched: true, DataFilters: [isnotnull(value_eur#459), (value_eur#459 > 1000000)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-co..., PartitionFilters: [], PushedFilters: [IsNotNull(value_eur), GreaterThan(value_eur,1000000)], ReadSchema: struct<value_eur:int>


time: 2 ms (started: 2026-07-27 21:06:11 +00:00)


In [25]:
df_select_filter.explain("formatted")

== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [value_eur#459]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parquet/male_players]
PushedFilters: [IsNotNull(value_eur), GreaterThan(value_eur,1000000)]
ReadSchema: struct<value_eur:int>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [value_eur#459]

(3) Filter [codegen id : 1]
Input [1]: [value_eur#459]
Condition : (isnotnull(value_eur#459) AND (value_eur#459 > 1000000))


time: 1.16 ms (started: 2026-07-27 21:06:11 +00:00)


In [26]:
df_select_filter.explain("extended")

== Parsed Logical Plan ==
'Project ['value_eur]
+- Filter (value_eur#459 > 1000000)
   +- Relation [player_id#449,player_url#450,fifa_version#451,fifa_update#452,fifa_update_date#453,short_name#454,long_name#455,player_positions#456,overall#457,potential#458,value_eur#459,wage_eur#460,age#461,dob#462,height_cm#463,weight_kg#464,league_id#465,league_name#466,league_level#467,club_team_id#468,club_name#469,club_position#470,club_jersey_number#471,club_loaned_from#472,club_joined_date#473,... 85 more fields] parquet

== Analyzed Logical Plan ==
value_eur: int
Project [value_eur#459]
+- Filter (value_eur#459 > 1000000)
   +- Relation [player_id#449,player_url#450,fifa_version#451,fifa_update#452,fifa_update_date#453,short_name#454,long_name#455,player_positions#456,overall#457,potential#458,value_eur#459,wage_eur#460,age#461,dob#462,height_cm#463,weight_kg#464,league_id#465,league_name#466,league_level#467,club_team_id#468,club_name#469,club_position#470,club_jersey_number#471,club_loaned_

In [27]:
df_select_filter.explain("codegen")

Found 1 WholeStageCodegen subtrees.
== Subtree 1 / 1 (maxMethodCodeSize:255; maxConstantPoolSize:140(0.21% used); numInnerClasses:0) ==
*(1) Filter (isnotnull(value_eur#459) AND (value_eur#459 > 1000000))
+- *(1) ColumnarToRow
   +- FileScan parquet [value_eur#459] Batched: true, DataFilters: [isnotnull(value_eur#459), (value_eur#459 > 1000000)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-co..., PartitionFilters: [], PushedFilters: [IsNotNull(value_eur), GreaterThan(value_eur,1000000)], ReadSchema: struct<value_eur:int>

Generated code:
/* 001 */ public Object generate(Object[] references) {
/* 002 */   return new GeneratedIteratorForCodegenStage1(references);
/* 003 */ }
/* 004 */
/* 005 */ // codegenStageId=1
/* 006 */ final class GeneratedIteratorForCodegenStage1 extends org.apache.spark.sql.execution.BufferedRowIterator {
/* 007 */   private Object[] references;
/* 008 */   private scala.collection.Itera

In [28]:
df_select_filter.explain("cost")

== Optimized Logical Plan ==
Project [value_eur#459], Statistics(sizeInBytes=12.7 MiB)
+- Filter (isnotnull(value_eur#459) AND (value_eur#459 > 1000000)), Statistics(sizeInBytes=1220.7 MiB)
   +- Relation [player_id#449,player_url#450,fifa_version#451,fifa_update#452,fifa_update_date#453,short_name#454,long_name#455,player_positions#456,overall#457,potential#458,value_eur#459,wage_eur#460,age#461,dob#462,height_cm#463,weight_kg#464,league_id#465,league_name#466,league_level#467,club_team_id#468,club_name#469,club_position#470,club_jersey_number#471,club_loaned_from#472,club_joined_date#473,... 85 more fields] parquet, Statistics(sizeInBytes=1220.7 MiB)

== Physical Plan ==
*(1) Filter (isnotnull(value_eur#459) AND (value_eur#459 > 1000000))
+- *(1) ColumnarToRow
   +- FileScan parquet [value_eur#459] Batched: true, DataFilters: [isnotnull(value_eur#459), (value_eur#459 > 1000000)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/kaggle/datasets/stefano

In [29]:
benchmark("Filtered, with predicate pushdown", df_select_filter)

Filtered, with predicate pushdown: 3.929 s
time: 8.34 s (started: 2026-07-27 21:06:11 +00:00)


In [30]:
%run_nb spark-show

Job Id ▾,Description,Submitted,Duration,Stages: Succeeded/Total,Tasks (for all stages): Succeeded/Total
18,collect at /tmp/ipykernel_8202/1013086114.py:6 collect at /tmp/ipykernel_8202/1013086114.py:6,2026/07/27 21:06:15,0.2 s,1/1,12/12
17,collect at /tmp/ipykernel_8202/1013086114.py:3 collect at /tmp/ipykernel_8202/1013086114.py:3,2026/07/27 21:06:11,0.2 s,1/1,12/12
16,collect at /tmp/ipykernel_8202/1013086114.py:6 collect at /tmp/ipykernel_8202/1013086114.py:6,2026/07/27 21:06:00,0.2 s,1/1,12/12
15,collect at /tmp/ipykernel_8202/1013086114.py:3 collect at /tmp/ipykernel_8202/1013086114.py:3,2026/07/27 21:05:49,0.4 s,1/1,12/12
14,$anonfun$withThreadLocalCaptured$2 at CompletableFuture.java:1768 $anonfun$withThreadLocalCaptured$2 at CompletableFuture.java:1768,2026/07/27 21:05:48,9 ms,1/1 (1 skipped),1/1 (12 skipped)
13,$anonfun$withThreadLocalCaptured$2 at CompletableFuture.java:1768 $anonfun$withThreadLocalCaptured$2 at CompletableFuture.java:1768,2026/07/27 21:05:48,51 ms,1/1,12/12
12,$anonfun$withThreadLocalCaptured$2 at CompletableFuture.java:1768 $anonfun$withThreadLocalCaptured$2 at CompletableFuture.java:1768,2026/07/27 21:05:48,19 ms,1/1 (1 skipped),1/1 (12 skipped)
11,$anonfun$withThreadLocalCaptured$2 at CompletableFuture.java:1768 $anonfun$withThreadLocalCaptured$2 at CompletableFuture.java:1768,2026/07/27 21:05:47,0.1 s,1/1,12/12
10,toPandas at /home/jovyan/work/spark/lib.py:66 toPandas at /home/jovyan/work/spark/lib.py:66,2026/07/27 21:05:44,24 ms,1/1,1/1
9,$anonfun$withThreadLocalCaptured$2 at CompletableFuture.java:1768 $anonfun$withThreadLocalCaptured$2 at CompletableFuture.java:1768,2026/07/27 21:05:43,12 ms,1/1 (1 skipped),1/1 (12 skipped)


Stage Id ▾,Description,Submitted,Duration,Tasks: Succeeded/Total,Input,Output,Shuffle Read,Shuffle Write
23,collect at /tmp/ipykernel_8202/1013086114.py:6 +details org.apache.spark.sql.classic.Dataset.collectToPython(Dataset.scala:2081) java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method) java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:75) java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:52) java.base/java.lang.reflect.Method.invoke(Method.java:580) py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244) py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374) py4j.Gateway.invoke(Gateway.java:282) py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132) py4j.commands.CallCommand.execute(CallCommand.java:79) py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184) py4j.ClientServerConnection.run(ClientServerConnection.java:108) java.base/java.lang.Thread.run(Thread.java:1583),2026/07/27 21:06:15,0.2 s,12/12,8.1 MiB,,,
22,collect at /tmp/ipykernel_8202/1013086114.py:3 +details org.apache.spark.sql.classic.Dataset.collectToPython(Dataset.scala:2081) java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method) java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:75) java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:52) java.base/java.lang.reflect.Method.invoke(Method.java:580) py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244) py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374) py4j.Gateway.invoke(Gateway.java:282) py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132) py4j.commands.CallCommand.execute(CallCommand.java:79) py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184) py4j.ClientServerConnection.run(ClientServerConnection.java:108) java.base/java.lang.Thread.run(Thread.java:1583),2026/07/27 21:06:11,0.2 s,12/12,8.1 MiB,,,
21,collect at /tmp/ipykernel_8202/1013086114.py:6 +details org.apache.spark.sql.classic.Dataset.collectToPython(Dataset.scala:2081) java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method) java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:75) java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:52) java.base/java.lang.reflect.Method.invoke(Method.java:580) py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244) py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374) py4j.Gateway.invoke(Gateway.java:282) py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132) py4j.commands.CallCommand.execute(CallCommand.java:79) py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184) py4j.ClientServerConnection.run(ClientServerConnection.java:108) java.base/java.lang.Thread.run(Thread.java:1583),2026/07/27 21:06:00,0.2 s,12/12,1137.5 KiB,,,
20,collect at /tmp/ipykernel_8202/1013086114.py:3 +details org.apache.spark.sql.classic.Dataset.collectToPython(Dataset.scala:2081) java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method) java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:75) java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:52) java.base/java.lang.reflect.Method.invoke(Method.java:580) py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244) py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374) py4j.Gateway.invoke(Gateway.java:282) py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132) py4j.commands.CallCommand.execute(CallCommand.java:79) py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184) py4j.ClientServerConnection.run(ClientServerConnection.java:108) 

Version,4.1.3
Master,local[2]
AppName,main


time: 149 ms (started: 2026-07-27 21:06:19 +00:00)
